In [0]:
from pyspark.sql.functions import *
from urllib.parse import urlparse
import requests
import re

In [0]:
job_portals = spark.table("main_catalogue.jobs.workday_portals")

In [0]:
def get_job_posting(api_url,facet, locations, key):
    headers = {
        "accept": "application/json",
        "accept-language": "en-US",
        "content-type": "application/json",
    }
    payload = {
                "appliedFacets": {
                    facet: locations
                },
                "limit": 20,
                "offset": 0,
                "searchText": key
    }
    response = requests.post(
                api_url,
                headers=headers,
                json=payload
    )
            
    jobPostings = response.json()["jobPostings"]
    return jobPostings

In [0]:
def get_days(postedOn):
    if "Today" in postedOn:
        return 0
    elif "Yesterday" in postedOn:
        return 1
    else:
        match = re.search(r'(\d+)', postedOn)
        return int(match.group(1)) if match else None

In [0]:
search_keys = ["Azure", "Data Engineer", "Databricks", "Big Data", "ETL", "Data Platform", "Pyspark", "SQL"]
jobs = []
for row in job_portals.collect():
    api_url = row["api_url"]
    hostname = urlparse(api_url).hostname
    company = hostname.split(".myworkdayjobs")[0].split(".")[0]
    print("company:", company)
    url = row["url"]
    for key in search_keys:
        try:
            jobPostings = get_job_posting(api_url, row["facet_location"], row["locations"], key)
            for job in jobPostings:
                d = {
                    "title": job.get("title"),
                    "externalPath": job.get("externalPath"),
                    "locationsText": job.get("locationsText"),
                    "postedOn": job.get("postedOn"),
                    "company": company,
                    "url": url.strip("/") + job.get("externalPath"),
                    "api_url": api_url[:-5].strip("/") + job.get("externalPath"),
                    "days": get_days(job.get("postedOn"))
                }
                jobs.append(d)
        except:
            print("Error:", api_url)
jobs_df = spark.createDataFrame(jobs)
display(jobs_df)

In [0]:
jobs_df.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f"raw_catalogue.jobs.workday")